# 03 - Evaluation: Real vs cGAN vs Diffusion
1. Visual comparison (Real vs cGAN vs Diffusion for all 10 digits)
2. FID between real MNIST test images and synthetic images
3. Conditioning accuracy and diversity
4. **Downstream utility:** a CNN trained **only on synthetic data**, tested on the **real** MNIST test set
5. Stand-outs: FID vs number of sampling steps, linear vs cosine schedule, multi-digit CAPTCHAs

> FID uses torchvision's ImageNet Inception-v3 pool features (values are comparable inside this project, not 1:1 with numbers published with the TF-Inception weights). If the weights cannot be downloaded, the notebook falls back to the feature space of a MNIST-trained CNN and tells you so.

In [ ]:
import os, sys, random
import numpy as np
import torch
import matplotlib.pyplot as plt

# Set QUICK_RUN=1 in the environment for a tiny smoke-test run (few samples / 1 epoch). Leave unset for the real run.
QUICK_RUN = os.environ.get("QUICK_RUN", "0") == "1"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| QUICK_RUN:", QUICK_RUN)
os.makedirs("checkpoints", exist_ok=True); os.makedirs("outputs", exist_ok=True)

In [ ]:
import time
from IPython.display import Markdown, display
from data_utils.dataloader import dataset_to_tensors
from model import Generator, ConditionalUNet
from training import sample_images, get_beta_schedule
from utils.checkpoint import load_checkpoint
from utils.metrics import (train_classifier, evaluate_classifier, compute_fid, conditioning_accuracy,
                           intra_class_diversity)
from utils.visualize import (compare_real_vs_generated, show_class_grid, make_captcha, show_captchas)

N_PER_CLASS = 20 if QUICK_RUN else 1000       # synthetic images per digit per model
BASE_CH = 16 if QUICK_RUN else 64             # must match notebook 02
SAMPLE_STEPS = 20 if QUICK_RUN else 250       # re-spaced diffusion sampling steps for bulk generation
CLF_EPOCHS = 1 if QUICK_RUN else 5

## 1. Load real data and a reference classifier
The reference CNN is trained on **real** MNIST and is used only (a) to measure *conditioning accuracy* of the generators and (b) as FID feature fallback. It is **not** the synthetic-only classifier of section 5.

In [ ]:
real_train_x, real_train_y = dataset_to_tensors(train=True)
real_test_x, real_test_y = dataset_to_tensors(train=False)
if QUICK_RUN:
    real_train_x, real_train_y = real_train_x[:3000], real_train_y[:3000]
    real_test_x, real_test_y = real_test_x[:1000], real_test_y[:1000]
ref_cnn = train_classifier(real_train_x, real_train_y, device, epochs=CLF_EPOCHS)
ref_acc = evaluate_classifier(ref_cnn, real_test_x, real_test_y, device)
print(f"Reference CNN (trained on REAL data) test accuracy: {ref_acc:.4f}")

## 2. Load models and generate the synthetic dataset

In [ ]:
# ---- cGAN ----
ck = torch.load("checkpoints/cgan_final.pt", map_location=device, weights_only=False)
LATENT_DIM = ck["config"]["latent_dim"]
G = Generator(latent_dim=LATENT_DIM).to(device)
load_checkpoint("checkpoints/cgan_final.pt", {"generator": G}, device=device); G.eval()

# ---- Diffusion (EMA weights) ----
def load_diffusion(path):
    net = ConditionalUNet(base_channels=BASE_CH).to(device)
    ck = load_checkpoint(path, {"ema_model": net}, device=device)
    net.eval()
    return net, get_beta_schedule(ck["config"]["schedule"], ck["config"]["timesteps"])

unet, betas = load_diffusion("checkpoints/diffusion_linear_final.pt")
print("diffusion T =", len(betas))

@torch.no_grad()
def generate_cgan(labels, batch=1000):
    out = []
    for i in range(0, len(labels), batch):
        l = labels[i:i + batch].to(device)
        out.append(G(torch.randn(len(l), LATENT_DIM, device=device), l).cpu())
    return torch.cat(out)

def generate_diffusion(labels, net=None, b=None, steps=None, batch=1000):
    net = net or unet; b = b if b is not None else betas
    return torch.cat([sample_images(net, labels[i:i + batch], b, device, steps or SAMPLE_STEPS).cpu()
                      for i in range(0, len(labels), batch)])

gen_labels = torch.arange(10).repeat_interleave(N_PER_CLASS)

t0 = time.time(); cgan_x = generate_cgan(gen_labels); t_cgan = time.time() - t0
t0 = time.time(); diff_x = generate_diffusion(gen_labels); t_diff = time.time() - t0
cgan_y = diff_y = gen_labels
print(f"cGAN     : {len(cgan_x)} images in {t_cgan:.1f}s  ({1000 * t_cgan / len(cgan_x):.2f}s / 1000 imgs)")
print(f"Diffusion: {len(diff_x)} images in {t_diff:.1f}s  ({1000 * t_diff / len(diff_x):.2f}s / 1000 imgs, {SAMPLE_STEPS} steps)")

## 3. Visual comparison: Real vs cGAN vs Diffusion (all 10 digits)

In [ ]:
compare_real_vs_generated(real_test_x, real_test_y, cgan_x, cgan_y, diff_x, diff_y,
                          digits=range(10), n_per_digit=8, save_path="outputs/03_real_vs_cgan_vs_diffusion.png")

## 4. Quantitative metrics: FID, conditioning, diversity

In [ ]:
real_ref = real_test_x
fid_cgan, backend = compute_fid(real_ref, cgan_x, device, feature_net=ref_cnn)
fid_diff, _ = compute_fid(real_ref, diff_x, device, feature_net=ref_cnn, backend=backend)
# lower bound: FID of *real training* images vs real test images
fid_real, _ = compute_fid(real_ref, real_train_x[:len(cgan_x)], device, feature_net=ref_cnn, backend=backend)

cond_cgan = conditioning_accuracy(ref_cnn, cgan_x, cgan_y, device)
cond_diff = conditioning_accuracy(ref_cnn, diff_x, diff_y, device)
div_real = intra_class_diversity(real_test_x, real_test_y)
div_cgan = intra_class_diversity(cgan_x, cgan_y)
div_diff = intra_class_diversity(diff_x, diff_y)

print(f"FID backend: {backend}")
print(f"{'':12s}{'FID (lower=better)':>22s}{'cond. acc':>12s}{'diversity':>12s}{'s/1000 imgs':>14s}")
print(f"{'Real (train)':12s}{fid_real:22.2f}{ref_acc:12.4f}{div_real:12.3f}{'-':>14s}")
print(f"{'cGAN':12s}{fid_cgan:22.2f}{cond_cgan:12.4f}{div_cgan:12.3f}{1000 * t_cgan / len(cgan_x):14.2f}")
print(f"{'Diffusion':12s}{fid_diff:22.2f}{cond_diff:12.4f}{div_diff:12.3f}{1000 * t_diff / len(diff_x):14.2f}")

* **FID** - distance between real and generated feature distributions (realism + diversity); lower is better.
* **Conditioning accuracy** - how often a real-data classifier recognises the *requested* digit in the generated image.
* **Diversity** - mean intra-class pairwise pixel distance; closest to the real value is best (much lower => mode collapse).

## 5. Downstream utility: CNN trained ONLY on synthetic data, tested on REAL MNIST test set

In [ ]:
synthetic_sets = {
    "cGAN only": (cgan_x, cgan_y),
    "Diffusion only": (diff_x, diff_y),
    "cGAN + Diffusion": (torch.cat([cgan_x, diff_x]), torch.cat([cgan_y, diff_y])),
}
downstream = {}
for name, (x, y) in synthetic_sets.items():
    print(f"\n== Training CNN on synthetic data: {name} ({len(x)} images) ==")
    clf = train_classifier(x, y, device, epochs=CLF_EPOCHS)               # ONLY synthetic images
    downstream[name] = evaluate_classifier(clf, real_test_x, real_test_y, device)   # REAL test set
    print(f"-> accuracy on REAL MNIST test set: {downstream[name]:.4f}")

print("\n" + "=" * 52)
for k, v in downstream.items():
    print(f"{k:20s} real-test accuracy = {v * 100:6.2f}%")
print(f"{'(reference: real data)':20s} real-test accuracy = {ref_acc * 100:6.2f}%")

In [ ]:
plt.figure(figsize=(6, 3.5))
names = list(downstream) + ["Real data (ref.)"]; vals = [v * 100 for v in downstream.values()] + [ref_acc * 100]
plt.bar(names, vals, color=["#4C72B0", "#DD8452", "#55A868", "#999999"]); plt.ylim(max(0, min(vals) - 5), 100.5)
for i, v in enumerate(vals): plt.text(i, v + 0.1, f"{v:.1f}%", ha="center")
plt.ylabel("accuracy on real MNIST test (%)"); plt.xticks(rotation=15); plt.tight_layout()
plt.savefig("outputs/03_downstream_accuracy.png", dpi=150); plt.show()

## 6. Automatic summary of the results above

In [ ]:
better = lambda a, b, na, nb, lower=True: na if ((a < b) if lower else (a > b)) else nb
display(Markdown(f"""
| Model | FID ({backend}) ↓ | Conditioning acc ↑ | Diversity (real = {div_real:.2f}) | Downstream acc on real test ↑ | s / 1000 images ↓ |
|---|---|---|---|---|---|
| cGAN | {fid_cgan:.2f} | {cond_cgan:.3f} | {div_cgan:.2f} | {downstream['cGAN only']:.3f} | {1000 * t_cgan / len(cgan_x):.2f} |
| Diffusion ({SAMPLE_STEPS} steps) | {fid_diff:.2f} | {cond_diff:.3f} | {div_diff:.2f} | {downstream['Diffusion only']:.3f} | {1000 * t_diff / len(diff_x):.2f} |

* Higher **fidelity** (lower FID): **{better(fid_cgan, fid_diff, 'cGAN', 'Diffusion')}**
* Closer to real **diversity**: **{better(abs(div_cgan - div_real), abs(div_diff - div_real), 'cGAN', 'Diffusion')}**
* Better **conditioning**: **{better(cond_cgan, cond_diff, 'cGAN', 'Diffusion', lower=False)}**
* Better **downstream utility**: **{better(downstream['cGAN only'], downstream['Diffusion only'], 'cGAN', 'Diffusion', lower=False)}**
* Faster **generation**: **{better(t_cgan, t_diff, 'cGAN', 'Diffusion')}**
"""))

## 7. Analysis

| Model | FID (Inception) ↓ | Conditioning acc. ↑ | Diversity (real = 17.58) | Downstream acc. on real test ↑ | s / 1000 images ↓ |
|---|---|---|---|---|---|
| cGAN | 9.70 | 0.990 | 17.64 | 95.47% | 0.03 |
| Diffusion (250 steps) | 2.91 | 0.983 | 17.60 | 96.81% | 72.30 |

Reference points: FID between real train and real test images = 1.76; a CNN trained on real data reaches 99.18%; a CNN trained on cGAN + Diffusion data together reaches 97.59%. (Numbers from the run submitted with this project; re-running gives slightly different values.)

**Fidelity.** Diffusion produced clearly higher fidelity: FID 2.91 vs 9.70 for the cGAN (lower is better), and it is much closer to the real-vs-real baseline of 1.76.

**Diversity.** Both models match the real data well on intra-class pixel distance (17.64 and 17.60 vs 17.58), so neither shows mode collapse in this metric. The metric is coarse, so the near-tie should not be over-interpreted. (An earlier cGAN version using BatchNorm collapsed to almost one image per class in eval mode; replacing BatchNorm with GroupNorm fixed this.)

**Conditioning.** Both follow the requested digit very reliably (0.990 vs 0.983; real-data classifier reference 0.992). The gap is small and comes from a single run, so it is not significant.

**Downstream utility.** A CNN trained only on synthetic data reached 95.47% (cGAN), 96.81% (Diffusion) and 97.59% (both combined) on the real MNIST test set, versus 99.18% for real data. Diffusion data is more useful than cGAN data, and combining both is best.

**Practical suitability.** The cGAN needs one forward pass per image (0.03 s per 1000 images vs 72.3 s for diffusion at 250 steps, about 2,400x slower). The step study shows FID drops quickly up to about 100 steps (5.69) and is almost flat from 250 steps (5.44) onward, so 100-250 steps are a sensible trade-off; with only 10 steps diffusion is worse than the cGAN. That study uses 2,000 samples, which biases FID upward, so compare values only within that plot.

**Noise schedule.** In this run the cosine schedule gave a worse FID (8.09) than the linear schedule (2.91) at 250 sampling steps. It was not tuned, so this is not a general statement about cosine schedules.

**Recommendation.** For a large CAPTCHA dataset where quality matters most, use diffusion with about 100-250 steps, or a mix of both models (best downstream accuracy). If throughput is the main constraint, the cGAN is the practical choice.

**Limitations.** Single training run per model (cGAN 50 epochs, diffusion 30); FID uses torchvision's ImageNet Inception weights, so absolute values differ from published numbers.

## 8. Stand-out: FID vs number of sampling steps (linear schedule)

In [ ]:
N_FID = 100 if QUICK_RUN else 2000                       # fewer samples than above to keep this study affordable
STEP_LIST = [s for s in ([5, 10, 20] if QUICK_RUN else [10, 25, 50, 100, 250, 500, 1000]) if s <= len(betas)]
lab_s = torch.arange(10).repeat_interleave(N_FID // 10)
fids_vs_steps = []
for K in STEP_LIST:
    x = generate_diffusion(lab_s, steps=K)
    f, _ = compute_fid(real_ref, x, device, feature_net=ref_cnn, backend=backend)
    fids_vs_steps.append(f); print(f"steps={K:5d}  FID={f:.2f}")
plt.figure(figsize=(6, 3.8)); plt.plot(STEP_LIST, fids_vs_steps, "o-", label=f"diffusion ({N_FID} samples)")
plt.axhline(fid_cgan, color="gray", ls="--", label="cGAN (full set)")
plt.xscale("log"); plt.xlabel("number of sampling steps"); plt.ylabel(f"FID ({backend})"); plt.grid(alpha=0.3); plt.legend()
plt.title("FID vs number of reverse-diffusion steps"); plt.tight_layout(); plt.savefig("outputs/03_fid_vs_steps.png", dpi=150); plt.show()
print("Note: FID computed from only", N_FID, "samples is biased upwards, compare values within this plot only.")

## 9. Stand-out: linear vs cosine schedule

In [ ]:
cos_path = "checkpoints/diffusion_cosine_final.pt"
if os.path.exists(cos_path):
    unet_cos, betas_cos = load_diffusion(cos_path)
    x_cos = generate_diffusion(gen_labels, net=unet_cos, b=betas_cos)
    fid_cos, _ = compute_fid(real_ref, x_cos, device, feature_net=ref_cnn, backend=backend)
    print(f"FID linear schedule: {fid_diff:.2f} | FID cosine schedule: {fid_cos:.2f}  ({SAMPLE_STEPS} sampling steps each, {backend} features)")
    show_class_grid(x_cos, gen_labels, per_class=10, title="Cosine-schedule diffusion samples", save_path="outputs/03_cosine_samples.png")
else:
    print("No cosine checkpoint found - run notebook 02 with TRAIN_COSINE = True.")

## 10. Stand-out: multi-digit CAPTCHA generator
Builds 4-digit CAPTCHAs by sampling every digit from the better model (lower FID), then applying random rotation / vertical jitter / spacing / distractor lines.

In [ ]:
best = "Diffusion" if fid_diff < fid_cgan else "cGAN"
print("Using best model by FID:", best)

@torch.no_grad()
def sample_digit(d):
    lab = torch.tensor([d])
    return (generate_diffusion(lab) if best == "Diffusion" else generate_cgan(lab))[0]

codes = ["4821", "7305", "1969", "2468"]
captchas = [make_captcha(c, sample_digit, seed=i) for i, c in enumerate(codes)]
show_captchas(captchas, titles=[f"label: {c}" for c in codes], save_path="outputs/03_captchas.png")

## Conclusion
* Both models generate class-conditioned handwritten digits; the tables above quantify realism (FID), diversity, conditioning and downstream utility.
* The cGAN is the cheaper generator for bulk production; the diffusion model offers stable training and strong coverage, with its sampling cost controllable through the number of steps.
* The combined synthetic dataset can be exported by saving `cgan_x/diff_x` (and labels) for the CAPTCHA classifier training pipeline.